# Traceprop-LLM -- Stage 2: storage-matched scope sweep + planted-example detection (Pythia-1B, L4)

Supersedes `exp35_logix_lds_colab.ipynb` (its `rank_mode`/reverse-match design didn't work as conceived -- see `docs/mlsys/SESSION_LOG.md` UPDATE 4/5/6). Both `exp31` and `exp35` now run LogIX at its own NATURAL settings (no rank override) and measure REAL on-disk bytes/example, then match Traceprop's `proj_dim` to that measured number -- Traceprop is always the one that grows or shrinks, never LogIX.

Both scripts also run a hard-assert gradient validation (cosine vs. direct autograd, same >=0.9999 standard Traceprop holds itself to, plus a uniform-scale check across modules) before trusting any measurement -- this already caught and fixed several real bugs on CPU; keep it ON for this GPU run too, since GPU numerics can differ from CPU and it's cheap to check once.

Run order:
1. **1-repeat GPU dry run** (~5 min) -- catches GPU-only failures (CUDA OOM, dtype mismatches) that CPU smoke tests can't, before committing to the full run.
2. **Scope sweep**: `exp31` (speed + measured bytes) and `exp35` (LDS, default + storage-matched) at tracked scope `{last-1, last-6, all}` on Pythia-1B/SST-2.
3. **Planted-example detection** (`exp37`): self-influence-based mislabel detection on the same model/data, at ~1% overhead.

n_subsets=100 for the sweep's LDS numbers (lower-powered than the 500 used for the paper's canonical §3.4 numbers -- flagged explicitly, raise if there's time budget left).

## Setup: pin versions, mount Drive, assert L4, clone repo

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert 'L4' in torch.cuda.get_device_name(0), (
    f"expected an L4, got {torch.cuda.get_device_name(0)} -- Table 1/2, the sweep, and this "
    f"session's runs are all L4-only; a different GPU would make the numbers incomparable. "
    f"Reconnect and request an L4 runtime."
)

!pip -q install "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2" "numpy<2" datasets scipy scikit-learn
!pip -q install --ignore-requires-python logix-ai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .
%cd /content/Traceprop/experiments

## Step 0: 1-repeat GPU dry run (do this before anything else)

Tiny settings, gradient validation ON, all three scripts. If any of these fail, fix it here -- minute 5, not hour 2.

In [ ]:
!python exp31_logix_comparison.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --steps 5 --repeats 1 --warmup 1 --pca_cov_steps 2 --track 1 \
    --out /tmp/dryrun_exp31.json --force

In [ ]:
!python exp35_logix_lds.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
    --n_train 32 --n_test 8 --n_subsets 2 --epochs 1 --batch 8 --track 1 --lora_init pca \
    --out /tmp/dryrun_exp35.json --force

In [ ]:
!python exp37_planted_detection.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
    --n_train 32 --plant_frac 0.1 --epochs 1 --batch 8 --track 1 \
    --out /tmp/dryrun_exp37.json --force

**If all three printed `gradient validation OK` (exp31/exp35) and finished without error, proceed. If anything failed, stop and fix it before running the real sweep below -- do not proceed on a partial dry-run pass.**

## Step 1: scope sweep -- exp31 (speed + measured bytes) at {last-1, last-6, all}

In [ ]:
for track, label in [(1, 'last1'), (6, 'last6'), (0, 'all')]:
    print(f'=== exp31 track={track} ({label}) ===')
    !python exp31_logix_comparison.py --backend hf --model EleutherAI/pythia-1b --device cuda \
        --steps 200 --repeats 20 --warmup 10 --pca_cov_steps 20 --track {track} \
        --out results/exp31_pythia1b_track{track}.json --force
    !cp results/exp31_pythia1b_track{track}.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 2: scope sweep -- exp35 (LDS, default + storage-matched) at {last-1, last-6, all}

n_subsets=100 (lower-powered than the paper's canonical 500 -- raise if time allows). This is the most expensive part of the sweep (subset retraining dominates).

In [ ]:
for track, label in [(1, 'last1'), (6, 'last6'), (0, 'all')]:
    print(f'=== exp35 track={track} ({label}) ===')
    !python exp35_logix_lds.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
        --n_train 2000 --n_test 200 --n_subsets 100 --subset_frac 0.5 --epochs 3 \
        --batch 16 --proj_dim 512 --track {track} --lora_init pca \
        --out results/exp35_pythia1b_track{track}.json --force
    !cp results/exp35_pythia1b_track{track}.json results/exp35_pythia1b_track{track}_raw.npz \
        /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 3: planted-example detection (exp37)

In [ ]:
!python exp37_planted_detection.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
    --n_train 2000 --plant_frac 0.05 --epochs 3 --batch 16 --proj_dim 512 --track 1 \
    --out results/exp37_pythia1b_track1.json --force
!cp results/exp37_pythia1b_track1.json results/exp37_pythia1b_track1_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Read everything back

In [ ]:
import json, glob

print('=== exp31: overhead + measured bytes, by tracked scope ===')
for track, label in [(1, 'last1'), (6, 'last6'), (0, 'all')]:
    d = json.load(open(f'results/exp31_pythia1b_track{track}.json'))
    sm = d['storage_matching']
    print(f"  track={label}: tracked_modules={d['tracked_modules_count']} "
          f"measured_bytes/ex={sm['logix_bytes_per_example_measured']} "
          f"matching_proj_dim={sm['traceprop_proj_dim_to_match']}")
    for cfg_name, cfg in d['configs'].items():
        print(f"    {cfg_name}: overhead={cfg['overhead_pct_median']}% +/- {cfg['overhead_pct_std']}%"
              + (f" covariance_pass={cfg['covariance_pass_s']}s" if cfg.get('covariance_pass_s') else ''))
    gv = d['gradient_validation']
    print(f"    gradient_validation: worst_cosine={min(g['worst_cosine'] for g in gv):.6f} "
          f"worst_nonuniformity={max(g['scale_nonuniformity'] for g in gv):.6f}")

print('\n=== exp35: LDS, by tracked scope ===')
for track, label in [(1, 'last1'), (6, 'last6'), (0, 'all')]:
    d = json.load(open(f'results/exp35_pythia1b_track{track}.json'))
    sm = d['storage_matching']
    print(f"  track={label}: measured_bytes/ex={sm['logix_bytes_per_example_measured']} "
          f"matched_proj_dim={sm['traceprop_proj_dim_matched']}")
    for k, v in d['lds'].items():
        print(f"    {k:<22} {v['mean']:+.4f} +/- {v['std']:.4f}")
    gv = d['gradient_validation']
    print(f"    gradient_validation: worst_cosine={gv['worst_cosine']:.6f} "
          f"nonuniformity={gv['scale_nonuniformity']:.6f}")

print('\n=== exp37: planted-example detection ===')
d = json.load(open('results/exp37_pythia1b_track1.json'))
print(f"  overhead={d['overhead_pct']}% (inline capture during ordinary training)")
for k, v in d['detection'].items():
    print(f"  {k:<8} AUC={v['auc']:.4f} precision@{d['k_planted']}={v['precision_at_k']:.4f}")

## Bring results back to Claude Code

Paste the printed summary above. It will: (1) fill in the storage-matched §3.3 draft from `docs/mlsys/LOGIX_OUTCOME_DRAFTS.md` using the real numbers (that file needs updating for the new measured-bytes design -- the old rank-based table template is stale), (2) add the scope-sweep figure/table to main.tex (this also fills in the currently-pending "overhead vs. tracked parameters" claim), (3) write up the planted-detection result as new evidence for the paper's compliance/audit framing (EU AI Act angle already in the abstract).